In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load dataset
df = pd.read_csv("../data/credit_risk_dataset.csv")

print("Original dataset shape:", df.shape)

Original dataset shape: (32581, 12)


In [3]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)

Duplicate rows: 165
Dataset shape after removing duplicates: (32416, 12)


In [4]:
# Remove clearly invalid age values
df = df[(df["person_age"] >= 18) & (df["person_age"] <= 80)]

# Remove clearly invalid employment lengths
df = df[
    (df["person_emp_length"].isna()) |
    ((df["person_emp_length"] >= 0) & (df["person_emp_length"] <= 50))
]

# Income and loan amount must be positive
df = df[df["person_income"] > 0]
df = df[df["loan_amnt"] > 0]

print("Dataset shape after basic cleaning:", df.shape)

Dataset shape after basic cleaning: (32407, 12)


In [5]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True) * 100)

Features shape: (32407, 11)
Target shape: (32407,)

Target distribution:
loan_status
0    25319
1     7088
Name: count, dtype: int64

Target percentage:
loan_status
0   78.13
1   21.87
Name: proportion, dtype: float64


In [6]:
numerical_features = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

categorical_features = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file"
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']

Categorical features:
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training data: (25925, 11)
Testing data: (6482, 11)

Training target distribution:
loan_status
0   78.13
1   21.87
Name: proportion, dtype: float64

Testing target distribution:
loan_status
0   78.12
1   21.88
Name: proportion, dtype: float64


In [8]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [9]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [11]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (25925, 26)
Processed testing shape: (6482, 26)


In [12]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nFirst 20 processed features:")
print(feature_names[:20])

Number of processed features: 26

First 20 processed features:
['num__person_age' 'num__person_income' 'num__person_emp_length'
 'num__loan_amnt' 'num__loan_int_rate' 'num__loan_percent_income'
 'num__cb_person_cred_hist_length' 'cat__person_home_ownership_MORTGAGE'
 'cat__person_home_ownership_OTHER' 'cat__person_home_ownership_OWN'
 'cat__person_home_ownership_RENT' 'cat__loan_intent_DEBTCONSOLIDATION'
 'cat__loan_intent_EDUCATION' 'cat__loan_intent_HOMEIMPROVEMENT'
 'cat__loan_intent_MEDICAL' 'cat__loan_intent_PERSONAL'
 'cat__loan_intent_VENTURE' 'cat__loan_grade_A' 'cat__loan_grade_B'
 'cat__loan_grade_C']


In [13]:
print("Missing values before preprocessing:")

display(
    X_train.isnull().sum()[X_train.isnull().sum() > 0]
)

print("\nMissing values after preprocessing:")

print("Training missing values:", np.isnan(X_train_processed).sum())
print("Testing missing values:", np.isnan(X_test_processed).sum())

Missing values before preprocessing:


person_emp_length     713
loan_int_rate        2447
dtype: int64


Missing values after preprocessing:
Training missing values: 0
Testing missing values: 0


In [14]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_train_processed_df.head())

,num__person_age,num__person_income,num__person_emp_length,num__loan_amnt,num__loan_int_rate,num__loan_percent_income,num__cb_person_cred_hist_length,cat__person_home_ownership_MORTGAGE,cat__person_home_ownership_OTHER,cat__person_home_ownership_OWN,cat__person_home_ownership_RENT,cat__loan_intent_DEBTCONSOLIDATION,cat__loan_intent_EDUCATION,cat__loan_intent_HOMEIMPROVEMENT,cat__loan_intent_MEDICAL,cat__loan_intent_PERSONAL,cat__loan_intent_VENTURE,cat__loan_grade_A,cat__loan_grade_B,cat__loan_grade_C,cat__loan_grade_D,cat__loan_grade_E,cat__loan_grade_F,cat__loan_grade_G,cat__cb_person_default_on_file_N,cat__cb_person_default_on_file_Y
5393,-0.60,-0.41,0.81,-0.73,-1.70,-0.57,-0.94,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00
3438,-0.60,-0.57,-0.44,-1.13,-0.21,-0.94,-0.69,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00
20585,0.21,-0.60,-0.95,-0.73,-0.01,-0.28,0.80,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00
25269,0.21,0.24,1.57,2.45,1.40,1.40,0.05,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00
12202,-0.76,0.28,0.56,-0.89,1.36,-1.13,-0.69,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00


In [15]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print("Class weights:")
print(class_weight_dict)

Class weights:
{np.int64(0): np.float64(0.6399654406319427), np.int64(1): np.float64(2.2861552028218695)}


In [16]:
import joblib

joblib.dump(
    preprocessor,
    "../model/preprocessor.pkl"
)

print("Preprocessor saved successfully!")

Preprocessor saved successfully!


In [17]:
cleaned_data = df.copy()

cleaned_data.to_csv(
    "../data/credit_risk_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")
print("Shape:", cleaned_data.shape)

Cleaned dataset saved successfully!
Shape: (32407, 12)


In [18]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)

print("Original rows: 32,581")
print("Final rows:", len(df))
print("Final columns:", len(df.columns))

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nNumerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nProcessed features:", X_train_processed.shape[1])

print("\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nPreprocessing completed successfully.")

PREPROCESSING SUMMARY
Original rows: 32,581
Final rows: 32407
Final columns: 12

Training samples: 25925
Testing samples: 6482

Numerical features: 7
Categorical features: 4

Processed features: 26

Remaining missing values:
person_emp_length     887
loan_int_rate        3093
dtype: int64

Preprocessing completed successfully.
